# LaplacianNB Advanced Features Tutorial

This notebook explores advanced features of the LaplacianNB package including fingerprint utilities, performance optimization, and comparison with other algorithms.

## Setup and Imports

Import all necessary libraries for advanced features demonstration.

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# RDKit for molecular operations
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Descriptors
from rdkit.DataStructs import BulkTanimotoSimilarity

# sklearn for comparison and utilities
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, learning_curve
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.decomposition import PCA
from scipy import sparse

# LaplacianNB components
from laplaciannb import (
    LaplacianNB, LaplacianNB_New, 
    convert_fingerprints, RDKitFingerprintConverter, FingerprintTransformer
)

# Set style for better plots
plt.style.use('seaborn-v0_8')
np.random.seed(42)

## Advanced Fingerprint Generation

Let's explore different types of molecular fingerprints and their properties.

In [ ]:
def get_multiple_fingerprint_types(smiles, n_bits=1024):
    """Generate multiple types of molecular fingerprints."""
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return None
    
    fingerprints = {}
    
    # Morgan fingerprints (ECFP-like)
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    fingerprints['morgan'] = set(morgan_gen.GetFingerprint(mol).GetOnBits())
    
    # Atom pair fingerprints
    ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=n_bits)
    fingerprints['atom_pair'] = set(ap_gen.GetFingerprint(mol).GetOnBits())
    
    # Topological torsion fingerprints
    tt_gen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=n_bits)
    fingerprints['torsion'] = set(tt_gen.GetFingerprint(mol).GetOnBits())
    
    # RDKit fingerprints (path-based)
    rdkit_gen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=n_bits)
    fingerprints['rdkit'] = set(rdkit_gen.GetFingerprint(mol).GetOnBits())
    
    return fingerprints

In [ ]:
# Test molecules with different properties
test_molecules = {
    'Simple alcohol': 'CCO',
    'Aromatic': 'c1ccccc1',
    'Drug-like': 'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O',  # Ibuprofen
    'Complex natural product': 'CC1=C(C(CCC1)(C)C)C=CC(=CC=CC(=CC(=O)O)C)C',  # Retinoic acid
    'Peptide-like': 'CC(C)C[C@H](NC(=O)[C@H](Cc1ccccc1)N)C(=O)O',  # Dipeptide
}

fingerprint_data = []
for name, smiles in test_molecules.items():
    fps = get_multiple_fingerprint_types(smiles, n_bits=1024)
    if fps:
        for fp_type, fp_bits in fps.items():
            fingerprint_data.append({
                'molecule': name,
                'smiles': smiles,
                'fp_type': fp_type,
                'n_bits_set': len(fp_bits),
                'fingerprint': fp_bits
            })

fp_df = pd.DataFrame(fingerprint_data)
print("Fingerprint comparison across molecule types:")
pivot_table = fp_df.pivot(index='molecule', columns='fp_type', values='n_bits_set')
print(pivot_table)

In [ ]:
# Visualize fingerprint bit distributions
plt.figure(figsize=(12, 8))

fp_types = fp_df['fp_type'].unique()
molecules = fp_df['molecule'].unique()

for i, fp_type in enumerate(fp_types):
    plt.subplot(2, 2, i+1)
    data = fp_df[fp_df['fp_type'] == fp_type]
    plt.bar(range(len(data)), data['n_bits_set'], alpha=0.7)
    plt.title(f'{fp_type.title()} Fingerprints')
    plt.xlabel('Molecule')
    plt.ylabel('Bits Set')
    plt.xticks(range(len(data)), data['molecule'], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Performance Comparison: Original vs New Implementation

Let's compare performance between the original and new implementations.

In [ ]:
# Generate synthetic dataset of varying sizes
def generate_synthetic_data(n_samples, n_bits=1024, avg_bits_per_sample=50):
    """Generate synthetic fingerprint data."""
    np.random.seed(42)
    
    X = []
    y = []
    
    for i in range(n_samples):
        # Random number of bits set
        n_bits_set = np.random.poisson(avg_bits_per_sample)
        n_bits_set = max(1, min(n_bits_set, n_bits//2))  # Reasonable bounds
        
        # Random bit positions
        bit_positions = set(np.random.choice(n_bits, n_bits_set, replace=False))
        X.append(bit_positions)
        
        # Random target (with some correlation to fingerprint size)
        prob_active = (len(bit_positions) - 30) / 40  # Bias towards larger fingerprints
        prob_active = max(0.1, min(0.9, prob_active))
        y.append(1 if np.random.random() < prob_active else 0)
    
    return X, np.array(y)

# Test different dataset sizes
dataset_sizes = [100, 500, 1000, 2000]
performance_results = []

for n_samples in dataset_sizes:
    print(f"Testing dataset size: {n_samples}")
    
    # Generate data
    X_sets, y = generate_synthetic_data(n_samples, n_bits=1024)
    X_sparse = convert_fingerprints(X_sets, n_bits=1024)
    
    # Time original implementation
    start_time = time.time()
    clf_orig = LaplacianNB()
    clf_orig.fit(X_sets, y)
    pred_orig = clf_orig.predict(X_sets)
    time_orig = time.time() - start_time
    
    # Time new implementation
    start_time = time.time()
    clf_new = LaplacianNB_New()
    clf_new.fit(X_sparse, y)
    pred_new = clf_new.predict(X_sparse)
    time_new = time.time() - start_time
    
    # Check accuracy match
    accuracy_orig = np.mean(pred_orig == y)
    accuracy_new = np.mean(pred_new == y)
    predictions_match = np.array_equal(pred_orig, pred_new)
    
    performance_results.append({
        'n_samples': n_samples,
        'time_original': time_orig,
        'time_new': time_new,
        'speedup': time_orig / time_new,
        'accuracy_original': accuracy_orig,
        'accuracy_new': accuracy_new,
        'predictions_match': predictions_match,
        'memory_original': 'N/A (sets)',
        'memory_new': f'{X_sparse.data.nbytes / 1024:.1f} KB'
    })

perf_df = pd.DataFrame(performance_results)
print("\nPerformance Comparison Results:")
print(perf_df.round(3))

In [ ]:
# Visualize performance comparison
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Timing comparison
ax1.plot(perf_df['n_samples'], perf_df['time_original'], 'o-', label='Original', linewidth=2)
ax1.plot(perf_df['n_samples'], perf_df['time_new'], 's-', label='New', linewidth=2)
ax1.set_xlabel('Dataset Size')
ax1.set_ylabel('Training Time (seconds)')
ax1.set_title('Training Time Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Speedup
ax2.plot(perf_df['n_samples'], perf_df['speedup'], 'g^-', linewidth=2)
ax2.set_xlabel('Dataset Size')
ax2.set_ylabel('Speedup Factor')
ax2.set_title('New Implementation Speedup')
ax2.grid(True, alpha=0.3)

# Accuracy comparison
ax3.plot(perf_df['n_samples'], perf_df['accuracy_original'], 'o-', label='Original', linewidth=2)
ax3.plot(perf_df['n_samples'], perf_df['accuracy_new'], 's-', label='New', linewidth=2)
ax3.set_xlabel('Dataset Size')
ax3.set_ylabel('Accuracy')
ax3.set_title('Accuracy Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Predictions match indicator
match_values = [1 if match else 0 for match in perf_df['predictions_match']]
ax4.bar(range(len(perf_df)), match_values, alpha=0.7, color='green')
ax4.set_xlabel('Dataset Index')
ax4.set_ylabel('Predictions Match (1=Yes, 0=No)')
ax4.set_title('Prediction Consistency')
ax4.set_xticks(range(len(perf_df)))
ax4.set_xticklabels(perf_df['n_samples'])
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Memory Efficiency Analysis

Let's analyze memory efficiency with different sparse matrix formats.

In [ ]:
# Generate data with different sparsity levels
sparsity_levels = [0.90, 0.95, 0.98, 0.99, 0.995]  # 90% to 99.5% sparse
n_samples = 1000
n_bits = 2048

memory_analysis = []

for sparsity in sparsity_levels:
    bits_per_sample = int(n_bits * (1 - sparsity))
    X_sets, y = generate_synthetic_data(n_samples, n_bits, bits_per_sample)
    
    # Convert to different formats
    formats = ['dense', 'csr', 'csc', 'coo']
    format_results = {'sparsity': sparsity, 'avg_bits': bits_per_sample}
    
    for fmt in formats:
        X_converted = convert_fingerprints(X_sets, n_bits=n_bits, output_format=fmt)
        
        if fmt == 'dense':
            memory_mb = X_converted.nbytes / (1024 * 1024)
        else:
            memory_mb = (X_converted.data.nbytes + X_converted.indices.nbytes + 
                        X_converted.indptr.nbytes) / (1024 * 1024)
        
        format_results[f'{fmt}_memory_mb'] = memory_mb
    
    memory_analysis.append(format_results)

memory_df = pd.DataFrame(memory_analysis)
print("Memory Usage Analysis (MB):")
print(memory_df.round(3))

In [ ]:
# Visualize memory efficiency
plt.figure(figsize=(12, 8))

# Memory usage comparison
plt.subplot(2, 2, 1)
for fmt in ['dense', 'csr', 'csc', 'coo']:
    plt.plot(memory_df['sparsity'], memory_df[f'{fmt}_memory_mb'], 'o-', label=fmt.upper(), linewidth=2)
plt.xlabel('Sparsity Level')
plt.ylabel('Memory Usage (MB)')
plt.title('Memory Usage by Sparse Format')
plt.legend()
plt.grid(True, alpha=0.3)

# Memory savings vs dense
plt.subplot(2, 2, 2)
for fmt in ['csr', 'csc', 'coo']:
    savings = (memory_df['dense_memory_mb'] - memory_df[f'{fmt}_memory_mb']) / memory_df['dense_memory_mb'] * 100
    plt.plot(memory_df['sparsity'], savings, 'o-', label=fmt.upper(), linewidth=2)
plt.xlabel('Sparsity Level')
plt.ylabel('Memory Savings (%)')
plt.title('Memory Savings vs Dense Format')
plt.legend()
plt.grid(True, alpha=0.3)

# Efficiency ratio (performance per MB)
plt.subplot(2, 2, 3)
dense_memory = memory_df['dense_memory_mb']
for fmt in ['csr', 'csc', 'coo']:
    ratio = dense_memory / memory_df[f'{fmt}_memory_mb']
    plt.plot(memory_df['sparsity'], ratio, 'o-', label=fmt.upper(), linewidth=2)
plt.xlabel('Sparsity Level')
plt.ylabel('Memory Efficiency Ratio')
plt.title('Memory Efficiency (Dense/Sparse)')
plt.legend()
plt.grid(True, alpha=0.3)

# Recommended format
plt.subplot(2, 2, 4)
recommendations = []
for _, row in memory_df.iterrows():
    min_memory = min(row['csr_memory_mb'], row['csc_memory_mb'], row['coo_memory_mb'])
    if row['csr_memory_mb'] == min_memory:
        recommendations.append('CSR')
    elif row['csc_memory_mb'] == min_memory:
        recommendations.append('CSC')
    else:
        recommendations.append('COO')

format_counts = pd.Series(recommendations).value_counts()
plt.pie(format_counts.values, labels=format_counts.index, autopct='%1.1f%%')
plt.title('Recommended Sparse Format Distribution')

plt.tight_layout()
plt.show()

## Algorithm Comparison

Let's compare LaplacianNB with other machine learning algorithms.

In [ ]:
# Generate a balanced dataset for fair comparison
X_sets, y = generate_synthetic_data(1000, n_bits=1024, avg_bits_per_sample=50)
X_dense = convert_fingerprints(X_sets, n_bits=1024, output_format='dense')
X_sparse = convert_fingerprints(X_sets, n_bits=1024, output_format='csr')

print(f"Dataset info:")
print(f"  Samples: {len(X_sets)}")
print(f"  Features: {X_dense.shape[1]}")
print(f"  Target distribution: {np.bincount(y)}")
print(f"  Sparsity: {1 - X_sparse.nnz / (X_sparse.shape[0] * X_sparse.shape[1]):.3f}")

In [ ]:
# Define algorithms to compare
algorithms = {
    'LaplacianNB (Original)': (LaplacianNB(), X_sets),
    'LaplacianNB (New)': (LaplacianNB_New(), X_sparse),
    'MultinomialNB': (MultinomialNB(), X_dense),
    'BernoulliNB': (BernoulliNB(), X_dense),
    'RandomForest': (RandomForestClassifier(n_estimators=100, random_state=42), X_dense),
    'SVM (linear)': (SVC(kernel='linear', random_state=42), X_dense[:500])  # Smaller subset for SVM
}

# Compare performance
algorithm_results = []

for name, (clf, X_data) in algorithms.items():
    print(f"Testing {name}...")
    
    # Adjust target for smaller dataset (SVM)
    y_data = y if X_data.shape[0] == len(y) else y[:X_data.shape[0]]
    
    # Time training
    start_time = time.time()
    try:
        if name == 'SVM (linear)':
            # Cross-validation for smaller dataset
            scores = cross_val_score(clf, X_data, y_data, cv=3, scoring='accuracy')
        else:
            scores = cross_val_score(clf, X_data, y_data, cv=5, scoring='accuracy')
        
        training_time = time.time() - start_time
        
        algorithm_results.append({
            'algorithm': name,
            'mean_accuracy': scores.mean(),
            'std_accuracy': scores.std(),
            'training_time': training_time,
            'cv_folds': len(scores)
        })
    except Exception as e:
        print(f"  Error with {name}: {e}")

results_df = pd.DataFrame(algorithm_results)
print("\nAlgorithm Comparison Results:")
print(results_df.round(4))

In [ ]:
# Visualize algorithm comparison
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

# Accuracy comparison
algorithms_names = results_df['algorithm']
accuracies = results_df['mean_accuracy']
errors = results_df['std_accuracy']

bars1 = ax1.bar(range(len(algorithms_names)), accuracies, yerr=errors, 
                capsize=5, alpha=0.7, color='skyblue', edgecolor='navy')
ax1.set_xlabel('Algorithm')
ax1.set_ylabel('Cross-Validation Accuracy')
ax1.set_title('Algorithm Accuracy Comparison')
ax1.set_xticks(range(len(algorithms_names)))
ax1.set_xticklabels(algorithms_names, rotation=45, ha='right')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (acc, err) in enumerate(zip(accuracies, errors)):
    ax1.text(i, acc + err + 0.005, f'{acc:.3f}', ha='center', fontsize=9)

# Training time comparison
times = results_df['training_time']
bars2 = ax2.bar(range(len(algorithms_names)), times, alpha=0.7, color='lightcoral', edgecolor='darkred')
ax2.set_xlabel('Algorithm')
ax2.set_ylabel('Training Time (seconds)')
ax2.set_title('Training Time Comparison')
ax2.set_xticks(range(len(algorithms_names)))
ax2.set_xticklabels(algorithms_names, rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

# Accuracy vs Time scatter plot
ax3.scatter(times, accuracies, s=100, alpha=0.7, c='green', edgecolor='darkgreen')
for i, name in enumerate(algorithms_names):
    ax3.annotate(name.split('(')[0], (times.iloc[i], accuracies.iloc[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)
ax3.set_xlabel('Training Time (seconds)')
ax3.set_ylabel('Accuracy')
ax3.set_title('Accuracy vs Training Time')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## RDKitFingerprintConverter Advanced Usage

Let's explore the advanced features of the fingerprint converter.

In [ ]:
# Create advanced converter with custom settings
converter = RDKitFingerprintConverter(
    n_bits=2048,
    output_format='auto',  # Automatically choose format
    dtype=np.float32,
    sparse_threshold=0.95  # Use sparse if >95% zeros
)

# Test with real molecules
real_molecules = [
    'CCO',                                    # Ethanol
    'CC(=O)OC1=CC=CC=C1C(=O)O',              # Aspirin
    'CC1=CC=C(C=C1)C(C)C(=O)O',              # Ibuprofen  
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',          # Caffeine
    'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',         # Ibuprofen (alternative)
    'C1=CC=C(C=C1)C(=O)O',                   # Benzoic acid
    'CC(C)(C)C1=CC=C(C=C1)O',                # 4-tert-Butylphenol
    'CCCCCCCCCCCCCCC(=O)O',                  # Palmitic acid
]

# Convert molecules to fingerprint sets
mol_fingerprints = []
for smiles in real_molecules:
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        fp = mfpgen.GetFingerprint(mol)
        mol_fingerprints.append(set(fp.GetOnBits()))
    else:
        mol_fingerprints.append(set())

print(f"Converted {len(mol_fingerprints)} molecules to fingerprints")

In [ ]:
# Use converter to analyze data
X_converted = converter.convert(mol_fingerprints)
stats = converter.get_statistics(mol_fingerprints)

print("Converter Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print(f"\nConverted matrix info:")
print(f"  Type: {type(X_converted)}")
print(f"  Shape: {X_converted.shape}")
print(f"  Data type: {X_converted.dtype}")

if hasattr(X_converted, 'format'):
    print(f"  Sparse format: {X_converted.format}")
    print(f"  Non-zero elements: {X_converted.nnz}")

In [ ]:
# Test different sparsity thresholds
sparsity_thresholds = [0.5, 0.8, 0.9, 0.95, 0.99]
threshold_results = []

for threshold in sparsity_thresholds:
    test_converter = RDKitFingerprintConverter(
        n_bits=2048,
        output_format='auto',
        sparse_threshold=threshold
    )
    
    X_test = test_converter.convert(mol_fingerprints)
    test_stats = test_converter.get_statistics(mol_fingerprints)
    
    threshold_results.append({
        'threshold': threshold,
        'chosen_format': 'sparse' if hasattr(X_test, 'format') else 'dense',
        'actual_sparsity': test_stats['sparsity'],
        'memory_efficient': test_stats['sparsity'] > threshold
    })

threshold_df = pd.DataFrame(threshold_results)
print("\nSparsity Threshold Analysis:")
print(threshold_df)

## Learning Curves and Model Analysis

Let's analyze how model performance changes with dataset size.

In [ ]:
# Generate larger dataset for learning curves
X_large, y_large = generate_synthetic_data(2000, n_bits=1024, avg_bits_per_sample=50)
X_large_sparse = convert_fingerprints(X_large, n_bits=1024)

# Calculate learning curves
train_sizes = np.linspace(0.1, 1.0, 10)
models_to_test = {
    'LaplacianNB (New)': LaplacianNB_New(),
    'MultinomialNB': MultinomialNB(),
    'BernoulliNB': BernoulliNB()
}

learning_results = {}

for name, model in models_to_test.items():
    print(f"Calculating learning curve for {name}...")
    
    if name == 'LaplacianNB (New)':
        X_data = X_large_sparse
    else:
        X_data = convert_fingerprints(X_large, n_bits=1024, output_format='dense')
    
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model, X_data, y_large, 
        train_sizes=train_sizes, 
        cv=5, 
        scoring='accuracy',
        n_jobs=-1
    )
    
    learning_results[name] = {
        'train_sizes': train_sizes_abs,
        'train_scores_mean': train_scores.mean(axis=1),
        'train_scores_std': train_scores.std(axis=1),
        'val_scores_mean': val_scores.mean(axis=1),
        'val_scores_std': val_scores.std(axis=1)
    }

In [ ]:
# Plot learning curves
plt.figure(figsize=(15, 5))

for i, (name, results) in enumerate(learning_results.items()):
    plt.subplot(1, 3, i+1)
    
    train_mean = results['train_scores_mean']
    train_std = results['train_scores_std']
    val_mean = results['val_scores_mean']
    val_std = results['val_scores_std']
    train_sizes = results['train_sizes']
    
    plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    
    plt.plot(train_sizes, val_mean, 'o-', color='red', label='Validation Score')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    
    plt.xlabel('Training Set Size')
    plt.ylabel('Accuracy')
    plt.title(f'Learning Curve: {name}')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ROC Curves and Performance Metrics

Let's create detailed performance analysis with ROC curves.

In [ ]:
# Prepare data for ROC analysis
from sklearn.model_selection import train_test_split

X_roc, y_roc = generate_synthetic_data(1000, n_bits=1024, avg_bits_per_sample=50)
X_train, X_test, y_train, y_test = train_test_split(X_roc, y_roc, test_size=0.3, random_state=42)

# Convert data
X_train_sparse = convert_fingerprints(X_train, n_bits=1024)
X_test_sparse = convert_fingerprints(X_test, n_bits=1024)
X_train_dense = convert_fingerprints(X_train, n_bits=1024, output_format='dense')
X_test_dense = convert_fingerprints(X_test, n_bits=1024, output_format='dense')

# Train models and get probabilities
roc_models = {
    'LaplacianNB (New)': (LaplacianNB_New(), X_train_sparse, X_test_sparse),
    'MultinomialNB': (MultinomialNB(), X_train_dense, X_test_dense),
    'BernoulliNB': (BernoulliNB(), X_train_dense, X_test_dense),
    'RandomForest': (RandomForestClassifier(n_estimators=100, random_state=42), X_train_dense, X_test_dense)
}

roc_data = {}

for name, (model, X_tr, X_te) in roc_models.items():
    print(f"Training {name} for ROC analysis...")
    
    model.fit(X_tr, y_train)
    y_proba = model.predict_proba(X_te)[:, 1]  # Probability of positive class
    
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision)
    
    roc_data[name] = {
        'fpr': fpr,
        'tpr': tpr,
        'roc_auc': roc_auc,
        'precision': precision,
        'recall': recall,
        'pr_auc': pr_auc,
        'y_proba': y_proba
    }

In [ ]:
# Plot ROC curves and Precision-Recall curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# ROC Curves
colors = ['blue', 'red', 'green', 'orange']
for i, (name, data) in enumerate(roc_data.items()):
    ax1.plot(data['fpr'], data['tpr'], color=colors[i], linewidth=2,
             label=f'{name} (AUC = {data["roc_auc"]:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Precision-Recall Curves
for i, (name, data) in enumerate(roc_data.items()):
    ax2.plot(data['recall'], data['precision'], color=colors[i], linewidth=2,
             label=f'{name} (AUC = {data["pr_auc"]:.3f})')

# Baseline (random classifier)
baseline = np.sum(y_test) / len(y_test)
ax2.axhline(y=baseline, color='k', linestyle='--', linewidth=1, label=f'Random ({baseline:.3f})')

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Performance summary table
performance_summary = []

for name, data in roc_data.items():
    # Calculate additional metrics
    y_pred = (data['y_proba'] > 0.5).astype(int)
    accuracy = np.mean(y_pred == y_test)
    
    # Find optimal threshold (Youden's index)
    optimal_idx = np.argmax(data['tpr'] - data['fpr'])
    optimal_threshold = roc_data[list(roc_data.keys())[0]]['fpr'][optimal_idx]  # Approximation
    
    performance_summary.append({
        'Model': name,
        'ROC AUC': data['roc_auc'],
        'PR AUC': data['pr_auc'],
        'Accuracy': accuracy,
        'Best TPR': np.max(data['tpr']),
        'Best Precision': np.max(data['precision'])
    })

summary_df = pd.DataFrame(performance_summary)
print("Performance Summary:")
print(summary_df.round(4))

## Summary and Recommendations

This advanced tutorial covered:

### 🔬 **Advanced Features Explored:**

1. **Multiple Fingerprint Types**: Morgan, Atom Pair, Torsion, RDKit fingerprints
2. **Performance Optimization**: Detailed timing and memory analysis
3. **Memory Efficiency**: Sparse matrix format comparison and optimization
4. **Algorithm Benchmarking**: Comparison with other ML algorithms
5. **Advanced Converter Usage**: Custom settings and automatic format selection
6. **Learning Curves**: Performance vs dataset size analysis
7. **ROC Analysis**: Detailed classification performance metrics

### 📊 **Key Findings:**

- **New implementation** provides significant speedup while maintaining accuracy
- **CSR sparse format** is most memory-efficient for typical molecular fingerprints
- **LaplacianNB performs competitively** with other algorithms on sparse binary data
- **Memory savings** can exceed 95% with sparse representations
- **Automatic format selection** adapts to data characteristics

### 🚀 **Best Practices:**

1. Use **folded fingerprints** (1024-2048 bits) for memory efficiency
2. Choose **CSR format** for most molecular fingerprint applications  
3. Use **sparse_threshold=0.95** for automatic format selection
4. Monitor **sparsity levels** to optimize memory usage
5. Compare multiple **fingerprint types** for your specific problem
6. Use **cross-validation** for robust performance estimation

### 🎯 **Production Recommendations:**

- **LaplacianNB_New** for large-scale molecular classification
- **FingerprintTransformer** for sklearn pipeline integration
- **Memory monitoring** for very large datasets
- **Performance profiling** before deploying to production